# Prompt Evaluation Pipeline

Practical implementation of an LLM evaluation pipeline
using the Claude API. Covers:
- Automated dataset generation
- Model-as-judge grading  
- Iterative prompt improvement
- Syntax validation

Applied to: ELLO — voice-first AI assistant for overwhelmed parents.

In [44]:
from google.colab import userdata
from anthropic import Anthropic

my_api_key = userdata.get('ANTHROPIC_API_KEY')
client = Anthropic(api_key=my_api_key)
model = "claude-haiku-4-5-20251001"

print("Ready ✅")

Ready ✅


In [ ]:
!pip install anthropic -q


In [137]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [138]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python" or "json" or "regex"
        "solution_criteria : "Key criteria for evaluating the solution"
    },

    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)





In [139]:
# Outside the function ↓
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, and hyphens only, cannot start or end with a hyphen)', 'format': 'python', 'solution_criteria': "Function should return True for valid bucket names like 'my-bucket-123' and False for invalid names like 'My-Bucket' or 'bucket-', handle edge cases for length and special characters"}, {'task': 'Create a JSON CloudFormation template snippet for an AWS IAM policy that grants read-only access to a specific S3 bucket', 'format': 'json', 'solution_criteria': 'JSON should be valid CloudFormation syntax, include proper IAM policy structure with Effect, Action, and Resource fields, specifically allow s3:GetObject and s3:ListBucket actions'}, {'task': 'Write a regular expression to match valid AWS EC2 instance IDs (format: i- followed by 17 alphanumeric characters)', 'format': 'regex', 'solution_criteria': "Regex should match valid instance IDs like 'i

In [141]:
with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [142]:
def run_prompt(test_case):
  """Merge the prompt and test case input, then return the result of the prompt."""
  prompt = f"""
Please Solve the following task

Task: {test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```code")
  output = chat(messages, stop_sequences=["```"])
  return output


In [143]:
def grade_by_model(test_case, output):
  eval_prompt = f"""
  You are an expert AWS Code Reviewer. Your task is to evaluate the following AI-generated solution.

  Original Task:
  <task>
  {test_case["task"]}
  </task>

  Solution to evaluate
  <solution>
  {output}
  </solution>

  Here is some criteria you should use to evaluate the solution:
  <criteria>
  {test_case["solution_criteria"]}
  </criteria>

  Output Format:
  Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

  Respond with JSON. Keep your response concise and direct.
  Example Response Shape:
  {{
      "strengths": string[],
      "weaknesses": string[],
      "reasoning": string,
      "score": number
  }}
  """
  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [144]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response,test_case):
  format = test_case["format"]
  if format == "json":
    return validate_json(response)
  elif format == "python":
    return validate_python(response)
  elif format == "regex":
    return validate_regex(response)

In [145]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    #score = 10
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    #strengths = model_grade["strengths"]
    #weaknesses = model_grade["weaknesses"]

    syntax_grade = grade_syntax(output, test_case)
    score = (model_score + syntax_grade) / 2



    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [146]:
def run_eval(dataset):
  """Loads the data set and runs run_test_case with each case"""
  results = []
  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)
  return results

In [147]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)


In [148]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef validate_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    Rules:\n    - Length: 3-63 characters\n    - Characters: lowercase letters, numbers, and hyphens only\n    - Cannot start or end with a hyphen\n    \"\"\"\n    if not isinstance(bucket_name, str):\n        return False\n    \n    pattern = r'^[a-z0-9]([a-z0-9\\-]{1,61}[a-z0-9])?$'\n    return bool(re.match(pattern, bucket_name)) and 3 <= len(bucket_name) <= 63\n\n\n# Test cases\ntest_cases = [\n    (\"my-bucket\", True),\n    (\"my-bucket-123\", True),\n    (\"abc\", True),\n    (\"a\", False),\n    (\"ab\", False),\n    (\"my_bucket\", False),\n    (\"-my-bucket\", False),\n    (\"my-bucket-\", False),\n    (\"MY-BUCKET\", False),\n    (\"my--bucket\", True),\n    (\"a\" * 63, True),\n    (\"a\" * 64, False),\n    (\"123-bucket\", True),\n    (\"bucket.name\", False),\n    (\"bucket@name\", False),\n]\n\nf

In [152]:
from statistics import mean

def run_eval(dataset):
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average Score: {average_score}")
  return results


In [153]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)


Average Score: 8.333333333333334


In [154]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef validate_s3_bucket_name(name: str) -> bool:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    Rules:\n    - 3-63 characters long\n    - Lowercase letters, numbers, and hyphens only\n    - Cannot start or end with a hyphen\n    \"\"\"\n    if not isinstance(name, str):\n        return False\n    \n    pattern = r'^[a-z0-9]([a-z0-9\\-]{1,61}[a-z0-9])?$'\n    \n    return bool(re.match(pattern, name)) and 3 <= len(name) <= 63\n\n\n# Test cases\ntest_cases = [\n    (\"valid-bucket-name\", True),\n    (\"mybucket\", True),\n    (\"my-bucket-123\", True),\n    (\"a\" * 63, True),\n    (\"ab\", False),\n    (\"a\" * 64, False),\n    (\"-invalid\", False),\n    (\"invalid-\", False),\n    (\"Invalid\", False),\n    (\"my_bucket\", False),\n    (\"my.bucket\", False),\n    (\"\", False),\n    (\"123\", True),\n]\n\nfor name, expected in test_cases:\n    result = validate_s3_bucket_name(name)\n    status = \"\u2713\